# 01 — Données

Téléchargement des prix de clôture ajustés (Yahoo Finance) des trois actifs du portefeuille.

| Actif | Ticker | Place, devise |
|---|---|---|
| TotalEnergies | `TTE.PA` | Euronext Paris, EUR |
| iShares Core S&P 500 (capitalisant) | `SXR8.DE` | Xetra, EUR (non couvert contre le dollar) |
| iShares Core € Govt Bond (distribuant) | `EUNH.DE` | Xetra, EUR |

**Reproductibilité.** Le fichier `data/prix.csv` utilisé pour les résultats du README est versionné.
Par défaut, ce notebook le relit au lieu de retélécharger : Yahoo peut modifier ses données a posteriori.
Passer `TELECHARGER = True` pour repartir de la source.

In [1]:
import sys
from pathlib import Path

# Racine du projet (le notebook est dans notebooks/)
RACINE = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(RACINE))
DATA, FIGURES = RACINE / "data", RACINE / "figures"
FIGURES.mkdir(exist_ok=True)

In [2]:
import pandas as pd

TICKERS = ["TTE.PA", "SXR8.DE", "EUNH.DE"]
DEBUT, FIN = "2016-01-01", "2026-10-01"   # yfinance exclut la date de fin : dernier jour = 30/09/2026
TELECHARGER = False

In [3]:
if TELECHARGER:
    import yfinance as yf

    # 1. Vérifier la devise : "TTE" seul serait la cotation new-yorkaise, en dollars
    for t in TICKERS:
        print(t, yf.Ticker(t).fast_info["currency"])

    # 2. Prix ajustés des dividendes : sans ajustement, chaque détachement ressemblerait à une perte
    brut = yf.download(TICKERS, start=DEBUT, end=FIN, auto_adjust=True)["Close"]
    brut.to_csv(DATA / "prix_bruts.csv")
else:
    brut = pd.read_csv(DATA / "prix_bruts.csv", index_col=0, parse_dates=True) \
        if (DATA / "prix_bruts.csv").exists() else None
    print("Téléchargement désactivé : on relit data/prix.csv")

Téléchargement désactivé : on relit data/prix.csv


## Contrôles

Les trois titres cotent sur deux places (Paris et Francfort) aux calendriers légèrement différents.
Le tableau est construit sur l'union des dates : un jour de fermeture d'une place laisse un trou.

In [4]:
if brut is not None:
    print(brut.shape)
    print(brut.isna().sum())
    # Deux trous isolés sur SXR8 alors qu'EUNH (même place) cote : trous de la source
    print(brut[brut["SXR8.DE"].isna() & brut["EUNH.DE"].notna()])

## Nettoyage

Les trous sont comblés par le **dernier prix connu** (`ffill`) : un titre non coté un jour donné est
valorisé à sa dernière cotation, et son rendement ce jour-là vaut 0. On ne fait jamais de `bfill`,
qui utiliserait le prix du lendemain (fuite d'information du futur).

In [5]:
if brut is not None:
    prix = brut.ffill()
    prix.to_csv(DATA / "prix.csv")
else:
    prix = pd.read_csv(DATA / "prix.csv", index_col=0, parse_dates=True)

print(prix.shape, "| valeurs manquantes :", int(prix.isna().sum().sum()))
prix.head()

(2752, 3) | valeurs manquantes : 0


,EUNH.DE,SXR8.DE,TTE.PA
Date,,,
2016-01-04,107.943169,172.050003,21.604946
2016-01-05,108.076721,174.649994,21.407696
2016-01-06,108.245880,172.910004,21.202444
2016-01-07,107.889763,168.570007,20.687973
2016-01-08,108.201355,164.919998,19.912270
